### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
### Read all the PDF inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(r"C:\Users\prate\OneDrive\Pictures\Desktop\RAG\data\CERA_Testing files")
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 11 PDF files to process

Processing: aadhar (1).pdf
  ✓ Loaded 1 pages

Processing: aadhar signatory.pdf
  ✓ Loaded 1 pages

Processing: address proof.pdf
  ✓ Loaded 1 pages

Processing: authority letter.pdf
  ✓ Loaded 1 pages

Processing: deed partnership.pdf
  ✓ Loaded 1 pages

Processing: gst declaration.pdf
  ✓ Loaded 1 pages

Processing: gst.pdf
  ✓ Loaded 1 pages

Processing: pan.pdf
  ✓ Loaded 1 pages

Processing: security cheque 1.pdf
  ✓ Loaded 1 pages

Processing: security cheque 2.pdf
  ✓ Loaded 1 pages

Processing: tcs certificate.pdf
  ✓ Loaded 1 pages

Total documents loaded: 11


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar (1).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar (1).pdf', 'file_type': 'pdf'}, page_content='AADHAR'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar signatory', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar signatory.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar signatory.pdf', 'file_type': 'pdf'}, page_content='Aadhar  \nsignatory'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'address proof', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\address proof.pdf'

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 11 documents into 11 chunks

Example chunk:
Content: AADHAR...
Metadata: {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar (1).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar (1).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar (1).pdf', 'file_type': 'pdf'}, page_content='AADHAR'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar signatory', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar signatory.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar signatory.pdf', 'file_type': 'pdf'}, page_content='Aadhar  \nsignatory'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'address proof', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\address proof.pdf'

### Embeddings and Vector Store DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import os

class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            # Avoid Windows symlink cache warning from huggingface_hub
            os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimensionality: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager 


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1878.38it/s]


Model loaded successfully. Embedding dimensionality: 384


### Vector Store 

In [ ]:
class VectorStore:
    """Manages vector storage and retrieval using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documnents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documnents
Existing documents in collection: 0


In [ ]:
chunks

[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar (1).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar (1).pdf', 'file_type': 'pdf'}, page_content='AADHAR'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aadhar signatory', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar signatory.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'aadhar signatory.pdf', 'file_type': 'pdf'}, page_content='Aadhar  \nsignatory'),
 Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'address proof', 'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\address proof.pdf'

In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for all chunks

embeddings = embedding_manager.generate_embeddings(texts)

### Store in the the vector database

vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 11 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.40it/s]

Generated embeddings with shape: (11, 384)
Adding 11 documents to vector store...


Successfully added 11 documents to vector store
Total documents in collection: 11


### Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is Aadhar")

Retrieving documents for query: 'What is Aadhar'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_6572805e_0',
  'content': 'AADHAR',
  'metadata': {'source_file': 'aadhar (1).pdf',
   'creator': 'PyPDF',
   'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar (1).pdf',
   'title': 'aadhar',
   'doc_index': 0,
   'creationdate': '',
   'producer': 'Skia/PDF m149 Google Docs Renderer',
   'page': 0,
   'page_label': '1',
   'total_pages': 1,
   'file_type': 'pdf',
   'content_length': 6},
  'similarity_score': 0.7802317142486572,
  'distance': 0.21976828575134277,
  'rank': 1},
 {'id': 'doc_3c3835d9_1',
  'content': 'Aadhar  \nsignatory',
  'metadata': {'source_file': 'aadhar signatory.pdf',
   'source': 'C:\\Users\\prate\\OneDrive\\Pictures\\Desktop\\RAG\\data\\CERA_Testing files\\aadhar signatory.pdf',
   'doc_index': 1,
   'page': 0,
   'creationdate': '',
   'creator': 'PyPDF',
   'producer': 'Skia/PDF m149 Google Docs Renderer',
   'file_type': 'pdf',
   'content_length': 18,
   'page_label': '1',
   'total_pages': 1,
   '

### Integration Vectordb Context pipelinewith LLM output

In [ ]:
### Simple RAG Pipeline with the Groq LLM
from langchain_groq import ChatGroq
